# 05. 운전자 수익성 분석

티머니 STIS 택시 운행 데이터(D012)를 활용하여 운전자별 수익성을 분석한다.

**분석 항목:**
1. DRIVER_ID별 집계 (매출, 건수, 거리)
2. 실차율 계산
3. 시간당 매출 추정
4. 시간대/요일별 시간당 매출 히트맵
5. 상위 10% vs 하위 10% 운전자 비교
6. 플랫폼 수수료 영향 분석
7. 운전자 등급 분류 (A/B/C/D)
8. 요약

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
# Mac 사용 시 아래 줄로 교체
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료')

In [ ]:
# === 메모리 최적화 유틸 ===
import gc, psutil, os

def mem_usage():
    """현재 RAM 사용량 출력"""
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'RAM: {gb:.1f} GB')

def optimize_dtypes(df, cat_cols=None):
    """DataFrame 메모리 최적화"""
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    if cat_cols:
        for col in cat_cols:
            if col in df.columns:
                df[col] = df[col].astype('category')
    return df

CHUNK_SIZE = 1_000_000  # D012 chunk 크기
D012_CAT_COLS = ['RIDE_A_CD', 'ALIGHT_A_CD', 'DRIVER_ID', 'TAXI_VEHC_ID', 'TRANSP_BIZR_ID']

mem_usage()

In [ ]:
# D012 chunk 로드 (메모리 최적화)
usecols = ['RIDE_DTIME', 'ALIGHT_DTIME', 'PAY_AMT', 'RIDE_DIST', 'VACNTV_DIST', 
           'DRIVER_ID', 'RIDE_A_CD', 'ALIGHT_A_CD', 'PLTF_FEE_AMT']
dtypes = {'RIDE_DTIME': str, 'ALIGHT_DTIME': str, 'PAY_AMT': 'int32', 
          'RIDE_DIST': 'int32', 'VACNTV_DIST': 'int32', 'PLTF_FEE_AMT': 'int32'}

driver_agg_list = []
hourly_revenue_list = []
total = 0

for chunk in pd.read_csv('./DC_TBYXD012.csv', usecols=usecols, dtype=dtypes, chunksize=CHUNK_SIZE):
    chunk['ride_dt'] = pd.to_datetime(chunk['RIDE_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    chunk['alight_dt'] = pd.to_datetime(chunk['ALIGHT_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    chunk = chunk.dropna(subset=['ride_dt', 'alight_dt'])
    chunk['duration_min'] = (chunk['alight_dt'] - chunk['ride_dt']).dt.total_seconds() / 60
    chunk = chunk[(chunk['duration_min'] > 0) & (chunk['duration_min'] <= 180)]
    chunk['hour'] = chunk['ride_dt'].dt.hour
    chunk['weekday'] = chunk['ride_dt'].dt.dayofweek
    total += len(chunk)
    
    # 운전자별 집계
    drv = chunk.groupby('DRIVER_ID').agg(
        total_revenue=('PAY_AMT', 'sum'),
        trip_count=('PAY_AMT', 'count'),
        avg_fare=('PAY_AMT', 'mean'),
        total_ride_dist=('RIDE_DIST', 'sum'),
        total_vacancy_dist=('VACNTV_DIST', 'sum'),
        total_duration=('duration_min', 'sum'),
        total_fee=('PLTF_FEE_AMT', 'sum')
    ).reset_index()
    driver_agg_list.append(drv)
    
    # 시간대/요일별 매출
    hr = chunk.groupby(['weekday', 'hour']).agg(
        revenue=('PAY_AMT', 'sum'),
        count=('PAY_AMT', 'count'),
        total_min=('duration_min', 'sum')
    ).reset_index()
    hourly_revenue_list.append(hr)
    
    del chunk, drv, hr
    gc.collect()

# 운전자별 합산
df_driver = pd.concat(driver_agg_list).groupby('DRIVER_ID').agg(
    total_revenue=('total_revenue', 'sum'),
    trip_count=('trip_count', 'sum'),
    total_ride_dist=('total_ride_dist', 'sum'),
    total_vacancy_dist=('total_vacancy_dist', 'sum'),
    total_duration=('total_duration', 'sum'),
    total_fee=('total_fee', 'sum')
).reset_index()
df_driver['avg_fare'] = df_driver['total_revenue'] / df_driver['trip_count']
df_driver['occupancy_rate'] = df_driver['total_ride_dist'] / (df_driver['total_ride_dist'] + df_driver['total_vacancy_dist'])
df_driver['revenue_per_hour'] = df_driver['total_revenue'] / (df_driver['total_duration'] / 60)

# 시간대/요일별 합산
hourly_rev = pd.concat(hourly_revenue_list).groupby(['weekday', 'hour']).agg(
    revenue=('revenue', 'sum'),
    count=('count', 'sum'),
    total_min=('total_min', 'sum')
).reset_index()
hourly_rev['revenue_per_hour'] = hourly_rev['revenue'] / (hourly_rev['total_min'] / 60)

del driver_agg_list, hourly_revenue_list
gc.collect()

print(f'전체 건수: {total:,}')
print(f'운전자 수: {len(df_driver):,}')
mem_usage()
df_driver.head()

In [ ]:
# 시간 파싱
df['RIDE_DTIME'] = pd.to_datetime(df['RIDE_DTIME'], errors='coerce')
df['ALIGHT_DTIME'] = pd.to_datetime(df['ALIGHT_DTIME'], errors='coerce')

# 운행시간 (분)
df['trip_min'] = (df['ALIGHT_DTIME'] - df['RIDE_DTIME']).dt.total_seconds() / 60
# 비정상 값 제거 (0분 이하 또는 180분 초과)
df.loc[(df['trip_min'] <= 0) | (df['trip_min'] > 180), 'trip_min'] = np.nan

# 파생 컬럼
df['hour'] = df['RIDE_DTIME'].dt.hour
df['dayofweek'] = df['RIDE_DTIME'].dt.dayofweek
df['day_name'] = df['RIDE_DTIME'].dt.day_name()

print(f"유효 운행시간 건수: {df['trip_min'].notna().sum():,} / {len(df):,}")
print(f"평균 운행시간: {df['trip_min'].mean():.1f}분")

## 1. DRIVER_ID별 집계

In [ ]:
driver = df.groupby('DRIVER_ID').agg(
    total_revenue=('PAY_AMT', 'sum'),
    trip_count=('PAY_AMT', 'count'),
    avg_fare=('PAY_AMT', 'mean'),
    total_ride_dist=('RIDE_DIST', 'sum'),
    total_vacant_dist=('VACNTV_DIST', 'sum'),
    total_trip_min=('trip_min', 'sum'),
    total_fee=('PLTF_FEE_AMT', 'sum'),
).reset_index()

print(f'고유 운전자 수: {len(driver):,}')
driver.describe()

## 2. 실차율 계산

In [ ]:
# 실차율 = 운행거리 / (운행거리 + 빈차거리)
driver['occupied_rate'] = driver['total_ride_dist'] / (
    driver['total_ride_dist'] + driver['total_vacant_dist']
)
# 분모 0인 경우 처리
driver['occupied_rate'] = driver['occupied_rate'].fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(driver['occupied_rate'], bins=30, color='steelblue', edgecolor='black', linewidth=0.5)
axes[0].axvline(driver['occupied_rate'].mean(), color='red', linestyle='--', label=f"평균: {driver['occupied_rate'].mean():.2%}")
axes[0].set_title('운전자별 실차율 분포')
axes[0].set_xlabel('실차율')
axes[0].set_ylabel('운전자 수')
axes[0].legend()

axes[1].scatter(driver['occupied_rate'], driver['total_revenue'], alpha=0.5, s=15, color='steelblue')
axes[1].set_title('실차율 vs 총 매출')
axes[1].set_xlabel('실차율')
axes[1].set_ylabel('총 매출 (원)')

plt.tight_layout()
plt.show()

print(f"평균 실차율: {driver['occupied_rate'].mean():.2%}")
print(f"중앙값 실차율: {driver['occupied_rate'].median():.2%}")

## 3. 시간당 매출 추정

In [ ]:
# 운전자별 시간당 매출 = 총 매출 / 총 운행시간(시)
driver['total_trip_hour'] = driver['total_trip_min'] / 60
driver['revenue_per_hour'] = driver['total_revenue'] / driver['total_trip_hour']
driver.loc[driver['total_trip_hour'] <= 0, 'revenue_per_hour'] = np.nan

fig, ax = plt.subplots(figsize=(10, 5))
valid_rph = driver['revenue_per_hour'].dropna()
# 이상치 제거 (1~99 백분위)
q1, q99 = valid_rph.quantile(0.01), valid_rph.quantile(0.99)
ax.hist(valid_rph[(valid_rph >= q1) & (valid_rph <= q99)], bins=40, color='coral', edgecolor='black', linewidth=0.5)
ax.axvline(valid_rph.median(), color='blue', linestyle='--', label=f"중앙값: {valid_rph.median():,.0f}원")
ax.set_title('운전자별 시간당 매출 분포')
ax.set_xlabel('시간당 매출 (원)')
ax.set_ylabel('운전자 수')
ax.legend()
plt.tight_layout()
plt.show()

print(f"시간당 매출 평균: {valid_rph.mean():,.0f}원")
print(f"시간당 매출 중앙값: {valid_rph.median():,.0f}원")

## 4. 시간대/요일별 시간당 매출 히트맵

In [ ]:
# 건별 시간당 매출 = 요금 / (운행시간/60)
df['trip_hour'] = df['trip_min'] / 60
df['rev_per_hour'] = df['PAY_AMT'] / df['trip_hour']
df.loc[df['trip_hour'] <= 0, 'rev_per_hour'] = np.nan

day_kr = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}

heatmap_data = df.groupby(['dayofweek', 'hour'])['rev_per_hour'].median().reset_index()
heatmap_pivot = heatmap_data.pivot_table(index='dayofweek', columns='hour', values='rev_per_hour')
heatmap_pivot.index = [day_kr.get(i, i) for i in heatmap_pivot.index]

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(heatmap_pivot, cmap='YlOrRd', ax=ax, fmt=',.0f', annot=True, linewidths=0.5,
            annot_kws={'fontsize': 7})
ax.set_title('시간대/요일별 시간당 매출 중앙값 (원)', fontsize=14)
ax.set_xlabel('시간')
ax.set_ylabel('요일')
plt.tight_layout()
plt.show()

## 5. 상위 10% vs 하위 10% 운전자 비교

In [ ]:
# 매출 기준 상위/하위 10%
q90 = driver['total_revenue'].quantile(0.9)
q10 = driver['total_revenue'].quantile(0.1)

top10 = driver[driver['total_revenue'] >= q90]
bot10 = driver[driver['total_revenue'] <= q10]

compare = pd.DataFrame({
    '상위 10%': [
        f"{top10['total_revenue'].mean():,.0f}",
        f"{top10['trip_count'].mean():.1f}",
        f"{top10['avg_fare'].mean():,.0f}",
        f"{top10['occupied_rate'].mean():.2%}",
        f"{top10['revenue_per_hour'].mean():,.0f}",
    ],
    '하위 10%': [
        f"{bot10['total_revenue'].mean():,.0f}",
        f"{bot10['trip_count'].mean():.1f}",
        f"{bot10['avg_fare'].mean():,.0f}",
        f"{bot10['occupied_rate'].mean():.2%}",
        f"{bot10['revenue_per_hour'].mean():,.0f}",
    ]
}, index=['평균 총 매출', '평균 건수', '평균 요금', '평균 실차율', '시간당 매출'])

print('=== 상위 10% vs 하위 10% 운전자 비교 ===')
compare

In [ ]:
# 선호 시간대 비교
top10_ids = set(top10['DRIVER_ID'])
bot10_ids = set(bot10['DRIVER_ID'])

df_top = df[df['DRIVER_ID'].isin(top10_ids)]
df_bot = df[df['DRIVER_ID'].isin(bot10_ids)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_top['hour'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black', linewidth=0.5)
axes[0].set_title('상위 10% 운전자 시간대 분포')
axes[0].set_xlabel('시간')
axes[0].set_ylabel('건수')

df_bot['hour'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral', edgecolor='black', linewidth=0.5)
axes[1].set_title('하위 10% 운전자 시간대 분포')
axes[1].set_xlabel('시간')
axes[1].set_ylabel('건수')

plt.tight_layout()
plt.show()

In [ ]:
# 선호 지역 비교 (승차 행정동 Top 10)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_area = df_top['RIDE_A_CD'].astype(str).value_counts().head(10)
bot_area = df_bot['RIDE_A_CD'].astype(str).value_counts().head(10)

top_area.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='black', linewidth=0.5)
axes[0].set_title('상위 10% 선호 승차지역 Top 10')
axes[0].set_xlabel('건수')
axes[0].invert_yaxis()

bot_area.plot(kind='barh', ax=axes[1], color='coral', edgecolor='black', linewidth=0.5)
axes[1].set_title('하위 10% 선호 승차지역 Top 10')
axes[1].set_xlabel('건수')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 6. 플랫폼 수수료 영향 분석

In [ ]:
# 운전자별 수수료율 = 총 수수료 / 총 매출
driver['fee_rate'] = driver['total_fee'] / driver['total_revenue']
driver.loc[driver['total_revenue'] == 0, 'fee_rate'] = 0

# 수수료 제외 순매출
driver['net_revenue'] = driver['total_revenue'] - driver['total_fee']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) 수수료율 분포
axes[0].hist(driver['fee_rate'].dropna(), bins=30, color='steelblue', edgecolor='black', linewidth=0.5)
axes[0].set_title('운전자별 수수료율 분포')
axes[0].set_xlabel('수수료율 (수수료/매출)')
axes[0].set_ylabel('운전자 수')

# (b) 수수료 vs 매출
axes[1].scatter(driver['total_revenue'], driver['total_fee'], alpha=0.4, s=15, color='steelblue')
axes[1].set_title('총 매출 vs 플랫폼 수수료')
axes[1].set_xlabel('총 매출 (원)')
axes[1].set_ylabel('총 수수료 (원)')

# (c) 수수료율 vs 시간당 순매출
driver['net_rev_per_hour'] = driver['net_revenue'] / driver['total_trip_hour']
driver.loc[driver['total_trip_hour'] <= 0, 'net_rev_per_hour'] = np.nan
axes[2].scatter(driver['fee_rate'], driver['net_rev_per_hour'], alpha=0.4, s=15, color='coral')
axes[2].set_title('수수료율 vs 시간당 순매출')
axes[2].set_xlabel('수수료율')
axes[2].set_ylabel('시간당 순매출 (원)')

plt.tight_layout()
plt.show()

print(f"평균 수수료율: {driver['fee_rate'].mean():.2%}")
print(f"수수료 총액: {driver['total_fee'].sum():,.0f}원")
print(f"순매출 총액: {driver['net_revenue'].sum():,.0f}원")

## 7. 운전자 등급 분류 (A/B/C/D)

In [ ]:
# 등급 기준: 매출 * 실차율 점수
# 정규화 후 곱
from sklearn.preprocessing import MinMaxScaler

# MinMaxScaler 없을 경우 대비 수동 정규화
try:
    scaler = MinMaxScaler()
    driver[['rev_norm', 'occ_norm']] = scaler.fit_transform(
        driver[['total_revenue', 'occupied_rate']].fillna(0)
    )
except ImportError:
    rev_min, rev_max = driver['total_revenue'].min(), driver['total_revenue'].max()
    occ_min, occ_max = driver['occupied_rate'].min(), driver['occupied_rate'].max()
    driver['rev_norm'] = (driver['total_revenue'] - rev_min) / (rev_max - rev_min + 1e-9)
    driver['occ_norm'] = (driver['occupied_rate'] - occ_min) / (occ_max - occ_min + 1e-9)

driver['score'] = driver['rev_norm'] * driver['occ_norm']

# 사분위 기준 등급
driver['grade'] = pd.qcut(driver['score'], q=4, labels=['D', 'C', 'B', 'A'])

# 등급별 통계
grade_stats = driver.groupby('grade', observed=True).agg(
    운전자수=('DRIVER_ID', 'count'),
    평균매출=('total_revenue', 'mean'),
    평균실차율=('occupied_rate', 'mean'),
    평균시간당매출=('revenue_per_hour', 'mean'),
    평균수수료율=('fee_rate', 'mean'),
).round(2)

print('=== 운전자 등급별 통계 ===')
grade_stats

In [ ]:
# 등급 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

grade_colors = {'A': '#2ecc71', 'B': '#3498db', 'C': '#f39c12', 'D': '#e74c3c'}
grade_counts = driver['grade'].value_counts().sort_index(ascending=False)
colors = [grade_colors[g] for g in grade_counts.index]

axes[0].bar(grade_counts.index, grade_counts.values, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('운전자 등급 분포')
axes[0].set_xlabel('등급')
axes[0].set_ylabel('운전자 수')
for i, v in enumerate(grade_counts.values):
    axes[0].text(i, v + 0.5, str(v), ha='center')

# 등급별 매출 박스플롯
grade_order = ['A', 'B', 'C', 'D']
data_for_box = [driver[driver['grade'] == g]['total_revenue'].values for g in grade_order]
bp = axes[1].boxplot(data_for_box, labels=grade_order, patch_artist=True)
for patch, g in zip(bp['boxes'], grade_order):
    patch.set_facecolor(grade_colors[g])
    patch.set_alpha(0.6)
axes[1].set_title('등급별 총 매출 분포')
axes[1].set_xlabel('등급')
axes[1].set_ylabel('총 매출 (원)')

plt.tight_layout()
plt.show()

## 8. 요약

In [ ]:
summary = {
    '총 운전자 수': f"{len(driver):,}",
    '총 통행 건수': f"{driver['trip_count'].sum():,}",
    '전체 매출': f"{driver['total_revenue'].sum():,.0f}원",
    '운전자당 평균 매출': f"{driver['total_revenue'].mean():,.0f}원",
    '평균 실차율': f"{driver['occupied_rate'].mean():.2%}",
    '시간당 매출 중앙값': f"{driver['revenue_per_hour'].median():,.0f}원",
    '평균 수수료율': f"{driver['fee_rate'].mean():.2%}",
    '전체 수수료 총액': f"{driver['total_fee'].sum():,.0f}원",
    'A등급 운전자 수': f"{(driver['grade'] == 'A').sum():,}",
    'D등급 운전자 수': f"{(driver['grade'] == 'D').sum():,}",
}

summary_df = pd.DataFrame(list(summary.items()), columns=['항목', '값'])
print('=== 운전자 수익성 분석 요약 ===')
summary_df